# Whisper usage method

In [ ]:
!pip install --upgrade pip
!pip install --upgrade transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


class WhisperTranscriber:
    def __init__(self, model_id="openai/whisper-large-v3"):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=self.torch_dtype, low_cpu_mem_usage=True, 
            use_safetensors=True
    )
        
        model.to(self.device)
        
        processor = AutoProcessor.from_pretrained(model_id)
        
        self.pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=self.torch_dtype,
        device=self.device,
        return_timestamps=True,
    )

    def get_whisper_transcription(self, audio_path):
        result = self.pipe(audio_path, 
                  generate_kwargs={
                      "task": "transcribe", 
                      "language": "ukrainian",
                      #"return_timestamps": True,
                  },
                 )
        
        transcription = result['text']
        return transcription
        
        

# Implementing diarization

In [ ]:
!pip install pyannote.audio "numpy<2.0"
!pip install pydub


In [4]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
import torch

class PyannoteDiarizer:
    def __init__(self, hf_token):
        try:
            self.pipeline = Pipeline.from_pretrained(
                "pyannote/speaker-diarization-3.1",
                use_auth_token=hf_token
            )
            self.device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
            self.pipeline.to(self.device)
        except Exception as e:
            self.pipeline = None

    def diarize(self, audio_path: str):
        if self.pipeline is None:
            return None

        try:
            with ProgressHook() as hook:
                diarization = self.pipeline(audio_path, hook=hook)
                return diarization
        except Exception as e:
            return None



In [ ]:
# my Huffing Face token
hf_token = ''
audio_path = '/kaggle/input/my-ye-my-poruch/my_ye_my_poruch.mp3'

diarizer = PyannoteDiarizer(hf_token)
diarization = diarizer.diarize(audio_path)

print(diarization)


## Merge Annotations

In [6]:
from pyannote.core import Annotation, Segment


def merge_diarization_result(diarization):
    """
    Merge diarization result from pyannote in order to join two or more replicas
    by the same speaker.
    """
    merged_annotation = Annotation()
    # get all speaker turns and sort them by start time
    turns = sorted(diarization.itertracks(yield_label=True), key=lambda turn: turn[0].start)
    
    if turns:
        current_speaker = turns[0][2]
        current_start = turns[0][0].start
        current_end = turns[0][0].end
    
        for segment, _, speaker in turns[1:]:
            if speaker == current_speaker:
                current_end = segment.end
            else:
                merged_segment = Segment(current_start, current_end)
                merged_annotation[merged_segment] = current_speaker
    
                current_speaker = speaker
                current_start = segment.start
                current_end = segment.end
    
        merged_segment = Segment(current_start, current_end)
        merged_annotation[merged_segment] = current_speaker
        
    return merged_annotation


def filter_short_segments(diarization, min_duration_ms=250):
    """
    Filter and remove very short segments (less than 250 milliseconds), because
    such segments may lead to errors while using Whisper model later in the pipeline.
    """
    cleaned_annotation = Annotation()
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        if segment.duration > (min_duration_ms / 1000.0):
            cleaned_annotation[segment] = speaker
    return cleaned_annotation


def clean_pyannote_output(diarization):
    """
    Use two functions: merge_diarization_result and filter_short_segments to properly 
    clean output from the pyannote.
    """
    diarization = filter_short_segments(diarization)
    diarization = merge_diarization_result(diarization)
    return diarization
    

In [7]:
def format_time(seconds_float):
    """
    Converts time in seconds to proper time in hours, minutes and seconds.
    """
    total_seconds = int(seconds_float)
    milliseconds = round((seconds_float - total_seconds) * 1000)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return hours, minutes, seconds, milliseconds

In [8]:
from pydub import AudioSegment

def cut_audio(
    input_file_path,
    output_file_path,
    start_h, start_min, start_sec, start_ms,
    end_h, end_min, end_sec, end_ms):
    
    audio = AudioSegment.from_file(input_file_path, format="mp3")

    start = ((start_h * 3600 + start_min * 60) + start_sec) * 1000 + start_ms
    end = ((end_h * 3600 + end_min * 60) + end_sec) * 1000 + end_ms

    mid_seconds = audio[start:end]
    mid_seconds.export(output_file_path, format="mp3")
    return None


#cut_audio('/kaggle/input/storozhowa-zastawa/storozhowa_zastawa.mp3',
         #'/kaggle/working/zastawa-first-40-min.mp3',
         #0, 0, 0, 0,
         #0, 39, 15, 0)

In [ ]:
print("Original Diarization:")
print(diarization)

print("\nMerged Diarization:")
merged_annotation = clean_pyannote_output(diarization)
print(merged_annotation)


In [10]:
# write diarization result to the file
output_file_path = '/kaggle/working/my_ye_my_poruch_diarization.txt'

with open(output_file_path, 'w', encoding='utf-8') as f:
    for segment, track_id, speaker in merged_annotation.itertracks(yield_label=True):
        start_hours, start_minutes, start_seconds, start_milliseconds = format_time(segment.start)
        end_hours, end_minutes, end_seconds, end_milliseconds = format_time(segment.end)
        
        f.write(f"[{start_hours:02d}:{start_minutes:02d}:{start_seconds:02d}.{start_milliseconds:03d} --> "
        f"{end_hours:02d}:{end_minutes:02d}:{end_seconds:02d}.{end_milliseconds:03d}] "
        f"{track_id} {speaker}\n")

In [11]:
import re
from pyannote.core import Annotation, Segment


def parse_time_to_seconds(time_str):
    """
    Converts a 'HH:MM:SS.ms' string to total seconds as a float.
    """
    h, m, s_ms = time_str.split(':')
    s, ms = s_ms.split('.')
    total_seconds = int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000
    return total_seconds

def load_diarization(file_path):
    """
    Reads a custom diarization text file and returns a pyannote Annotation object.
    """
    # to find: (start_time_str, end_time_str, speaker_label)
    pattern = re.compile(r"\[\s*(\d{2}:\d{2}:\d{2}\.\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}\.\d{3})\]\s*\w+\s*(SPEAKER_\d+)")

    annotation = Annotation()

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                start_str, end_str, speaker = match.groups()
                
                start_seconds = parse_time_to_seconds(start_str)
                end_seconds = parse_time_to_seconds(end_str)
                
                segment = Segment(start_seconds, end_seconds)
                
                annotation[segment] = speaker
                
    return annotation


# usage
#txt_file_path = '/kaggle/input/viddana-part-diarization-2/viddana_part_diarization.txt'

#loaded_annotation = load_diarization(txt_file_path)

#print(f'Successfully loaded and converted the file.')
#print(f'The object type is: {type(loaded_annotation)}')
#print(loaded_annotation)

In [12]:
#merged_annotation = clean_pyannote_output(loaded_annotation)
#print(merged_annotation)

In [ ]:
# 1) iterate over merged annotation: take end and start of each speaker replica;
# 2) cut audio with given end and start timestamp
# 3) give cut audio to the Whisper model to get transcription
# 4) join speaker and their speech
# 5) write final transcription to the .txt and .json files
import json

transcriber = WhisperTranscriber(model_id="openai/whisper-large-v3")

# FULL AUDIO
#audio_path = '/kaggle/input/storozhowa-zastawa/storozhowa_zastawa.mp3'
final_transcription_file_path = '/kaggle/working/my_ye_my_poruch_full_transcription.txt'

for segment, _, speaker in merged_annotation.itertracks(yield_label=True):    
    start_h, start_m, start_s, start_ms = format_time(segment.start)
    end_h, end_m, end_s, end_ms = format_time(segment.end)
    
    start_timecode = f'{start_h}:{start_m}:{start_s}.{start_ms}'
    end_timecode = f'{end_h}:{end_m}:{end_s}.{end_ms}'

    print(f'Speaker: {speaker}')
    print(f'Start: {start_timecode}')
    print(f'End: {end_timecode}')
    # cut audio
    output_cut_audio = '/kaggle/working/cut.mp3'
    cut_audio(audio_path, output_cut_audio, start_h, start_m, 
                               start_s, start_ms, end_h, end_m, end_s, end_ms)
    # give cut audio to Whisper
    speech = transcriber.get_whisper_transcription(output_cut_audio)
    print(f'Speech: {speech}\n')

    
    

    # write to plain txt file
    with open(final_transcription_file_path, 'a', encoding='utf-8') as file:
        file.write(f'{speaker} | {start_timecode} --> {end_timecode} | {speech}\n')
    

In [15]:
#!rm -rf /kaggle/working/*